# Practice — Линейная регрессия (California Housing)

12 связанных упражнений на датасете **California Housing** (предсказываем медианную стоимость дома по характеристикам района Калифорнии). Полный ML-цикл: EDA → split → fit → predict → метрики → polynomial → CV → save.

Формат упражнения: **Что делаем / Вход / Ожидаемый выход**.

Если застрял — посмотри `02_linreg_sklearn_solution.ipynb`.


## О датасете

Используем **California Housing** из `sklearn.datasets.fetch_california_housing()` — 20 640 районов Калифорнии (по данным переписи 1990 года). Каждая строка — район; цель — медианная стоимость дома (`MedHouseVal`, в сотнях тысяч долларов).

| Колонка | Тип | Описание |
|---|---|---|
| `MedInc` | float | Медианный доход в районе, десятки тыс. долл. |
| `HouseAge` | float | Медианный возраст дома, годы |
| `AveRooms` | float | Среднее число комнат на дом |
| `AveBedrms` | float | Среднее число спален на дом |
| `Population` | float | Население района |
| `AveOccup` | float | Среднее число жильцов в доме |
| `Latitude` | float | Широта района |
| `Longitude` | float | Долгота района |
| `MedHouseVal` | float | **Целевая** — медианная цена дома, сотни тыс. долл. |

Размер: 20 640 строк × 9 колонок.

Бизнес-задача: модель оценщика недвижимости, которая по характеристикам района прикидывает справедливую цену.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
import joblib

RANDOM_STATE = 42
data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.shape)
df.head()


## Упражнение 1 — EDA: корреляции

**Что делаем:** посчитай корреляцию всех числовых колонок с целевой `MedHouseVal`. Округли до 2 знаков, отсортируй по убыванию модуля.

**Вход:** `df`.

**Ожидаемый выход:** Series длины 9 (включая саму `MedHouseVal=1.0`); самая сильная — `MedInc`.


In [ ]:
# твой код


## Упражнение 2 — Heatmap

**Что делаем:** построй `sns.heatmap` по `df.corr()`. Аннотируй значениями (`annot=True`), сделай цветовую шкалу `coolwarm`, заголовок «Корреляции California Housing».

**Вход:** `df`.

**Ожидаемый выход:** тепловая карта 9×9.


In [ ]:
# твой код


## Упражнение 3 — Split 80/20

**Что делаем:** раздели `df` на `X` (все фичи кроме `MedHouseVal`) и `y` (`MedHouseVal`). Через `train_test_split` с `test_size=0.2, random_state=42` сделай train/test. Выведи формы.

**Вход:** `df`.

**Ожидаемый выход:** `X_train` ~ 16 512 строк, `X_test` ~ 4 128 строк.


In [ ]:
# твой код
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']


## Упражнение 4 — Обучить baseline LinearRegression

**Что делаем:** обучи `LinearRegression` на `X_train, y_train`. Это будет baseline.

**Вход:** `X_train, y_train`.

**Ожидаемый выход:** обученная модель в переменной `model`.


In [ ]:
# твой код
model = ...


## Упражнение 5 — Коэффициенты и интерпретация

**Что делаем:** построй `pd.Series` коэффициентов с именами фичей, отсортируй по убыванию модуля, выведи. В комментарии напиши, какая фича влияет сильнее всего и какой знак.

**Вход:** `model, X.columns`.

**Ожидаемый выход:** Series длины 8 + строка-комментарий.


In [ ]:
# твой код
# вывод: ...


## Упражнение 6 — Все 4 метрики на test

**Что делаем:** на `X_test, y_test` посчитай MAE, RMSE, MAPE, R². Округли до 3 знаков, выведи аккуратно.

**Вход:** `model, X_test, y_test`.

**Ожидаемый выход:** 4 числа.


In [ ]:
# твой код


## Упражнение 7 — График y_true vs y_pred

**Что делаем:** на test построй scatter `y_true` vs `y_pred` + диагональ y=x красным. Подпиши оси, заголовок.

**Вход:** `y_test, y_pred`.

**Ожидаемый выход:** график.


In [ ]:
# твой код


## Упражнение 8 — Остатки

**Что делаем:** посчитай `residuals = y_test - y_pred`. Построй гистограмму остатков (30 бинов). Среднее остатков должно быть около 0.

**Вход:** `y_test, y_pred`.

**Ожидаемый выход:** график + строка с `residuals.mean()`.


In [ ]:
# твой код


## Упражнение 9 — Polynomial degree=2

**Что делаем:** собери Pipeline `[StandardScaler, PolynomialFeatures(2, include_bias=False), LinearRegression]`. Обучи на train, посчитай RMSE на test. Сравни с baseline из упр. 6.

**Вход:** `X_train, y_train, X_test, y_test`.

**Ожидаемый выход:** одно число — RMSE poly(2).


In [ ]:
# твой код
pipe2 = ...


## Упражнение 10 — K-fold сравнение трёх моделей

**Что делаем:** через `KFold(n_splits=5, shuffle=True, random_state=42)` и `cross_val_score(scoring='neg_root_mean_squared_error')` сравни три модели:
- baseline `LinearRegression`
- `Pipeline(StandardScaler, Poly(2), LinearRegression)`
- `Pipeline(StandardScaler, Poly(3), LinearRegression)`

Выведи mean ± std RMSE для каждой.

**Вход:** `X, y`.

**Ожидаемый выход:** 3 строки вида `baseline: 0.728 ± 0.012`.


In [ ]:
# твой код


## Упражнение 11 — Выбор и финальное обучение

**Что делаем:** выбери лучшую модель из упр. 10 (по mean RMSE при адекватном std), обучи её на ВСЕХ данных (`X, y`), сохрани через `joblib.dump` в `/tmp/my_california.pkl`.

**Вход:** `X, y`.

**Ожидаемый выход:** файл на диске.


In [ ]:
# твой код


## Упражнение 12 — Загрузить и предсказать новый район

**Что делаем:** загрузи модель из файла. Создай новый «район» как `pd.DataFrame` с одной строкой:

```python
new = pd.DataFrame([{
    'MedInc': 5.0, 'HouseAge': 25.0, 'AveRooms': 6.0, 'AveBedrms': 1.0,
    'Population': 1500.0, 'AveOccup': 3.0, 'Latitude': 34.0, 'Longitude': -118.0,
}])
```

Сделай `predict`, выведи цену в долларах (умножь на 100 000).

**Вход:** `joblib.load(...)`, `new`.

**Ожидаемый выход:** одно число (примерно 200 000 — 350 000 долл.).


In [ ]:
# твой код
